# Milestone 3: Calibration, Monotonic Constraints, and Segment Models

Three production-credit-risk techniques applied to the Milestone 2 models:

1. **Calibration** (Section 3A) — make predicted probabilities actually match observed default rates at every score level, not just on average. Necessary for pricing and EL calculations.
2. **Monotonic constraints** (Section 3B) — enforce domain-defensible relationships between key features and predicted risk. Required for regulator-friendly documentation.
3. **Segment models** (Section 3C) — train and evaluate a separate model on the thin-file segment to determine whether segment-specific modeling is justified.

This notebook builds directly on Milestone 2's data prep and trained models.

## Setup

Load the data, retrieve the cached train/val/test split, refit the Milestone 2 baseline (WoE+LR) and main model (LightGBM) on the training set, and generate raw validation/test predictions.

Re-fitting in this notebook (rather than loading saved Milestone 2 model artifacts) keeps the workflow self-contained and avoids version-skew between notebooks.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_joined, basic_clean
from src.splits import get_splits
from src.features import prepare_features, get_feature_columns

# Standard reload
df = load_joined()
df = basic_clean(df)
train_idx, val_idx, test_idx = get_splits(df)

df_features = prepare_features(df, include_ext_source_1=True)
train = df_features.loc[train_idx]
val = df_features.loc[val_idx]
test = df_features.loc[test_idx]

feature_cols = get_feature_columns(include_ext_source_1=True)
categorical_cols = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE",
]

X_train, y_train = train[feature_cols], train["TARGET"]
X_val, y_val = val[feature_cols], val["TARGET"]
X_test, y_test = test[feature_cols], test["TARGET"]

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

In [ ]:
from optbinning import BinningProcess
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

# --- WoE + LR baseline ---
binning_process = BinningProcess(
    variable_names=feature_cols,
    categorical_variables=categorical_cols,
    min_n_bins=3, max_n_bins=8, min_bin_size=0.05,
)
binning_process.fit(X_train, y_train)

X_train_woe = binning_process.transform(X_train)
X_val_woe = binning_process.transform(X_val)
X_test_woe = binning_process.transform(X_test)

lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_woe, y_train)

# --- LightGBM ---
X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()
X_test_lgb = X_test.copy()
for col in categorical_cols:
    X_train_lgb[col] = X_train_lgb[col].astype("category")
    X_val_lgb[col] = X_val_lgb[col].astype("category")
    X_test_lgb[col] = X_test_lgb[col].astype("category")

lgb_params = {
    "objective": "binary", "metric": "auc",
    "learning_rate": 0.05, "num_leaves": 63, "max_depth": -1,
    "min_data_in_leaf": 100, "feature_fraction": 0.8,
    "bagging_fraction": 0.8, "bagging_freq": 5,
    "random_state": 42, "verbose": -1,
}
lgb_train_ds = lgb.Dataset(X_train_lgb, label=y_train, categorical_feature=categorical_cols)
lgb_val_ds = lgb.Dataset(X_val_lgb, label=y_val, categorical_feature=categorical_cols, reference=lgb_train_ds)

lgb_model = lgb.train(
    lgb_params, lgb_train_ds, num_boost_round=2000,
    valid_sets=[lgb_train_ds, lgb_val_ds], valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=0)],
)

# Get predictions on val and test
y_val_pred_lr = lr_model.predict_proba(X_val_woe)[:, 1]
y_test_pred_lr = lr_model.predict_proba(X_test_woe)[:, 1]
y_val_pred_lgb = lgb_model.predict(X_val_lgb, num_iteration=lgb_model.best_iteration)
y_test_pred_lgb = lgb_model.predict(X_test_lgb, num_iteration=lgb_model.best_iteration)

print(f"LR best iteration: N/A (not boosted)")
print(f"LightGBM best iteration: {lgb_model.best_iteration}")

## Section 3A: Calibration

A model is well-calibrated if, among all applicants assigned probability p, the actual default rate is p. Mean calibration (predicted PD ≈ actual default rate on average) is necessary but not sufficient — a model could be mean-calibrated but systematically off at specific score levels.

Gradient-boosted models can be miscalibrated in subtle ways — class weighting in particular tends to push predictions toward extremes. We quantify any miscalibration with a reliability diagram and Expected Calibration Error (ECE), then apply isotonic regression to the LightGBM predictions to ensure they're usable for pricing and threshold-based decisions in Milestone 4.

The calibrator is fit on validation predictions (not training), since the model has already overfit slightly to training and its training-set scores are systematically too confident.

In [ ]:
def expected_calibration_error(y_true, y_pred, n_bins=10):
    """
    Expected Calibration Error: weighted average gap between predicted and actual
    default rates across n_bins quantile-based score bins.

    Lower is better. ECE = 0 means perfectly calibrated.
    """
    # Equal-frequency bins (each contains the same number of observations)
    bin_edges = np.quantile(y_pred, np.linspace(0, 1, n_bins + 1))
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    bin_ids = np.digitize(y_pred, bin_edges[1:-1])

    total = len(y_true)
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        bin_pred = y_pred[mask].mean()
        bin_actual = y_true[mask].mean()
        bin_weight = mask.sum() / total
        ece += bin_weight * abs(bin_pred - bin_actual)

    return ece


ece_lr = expected_calibration_error(y_val.values, y_val_pred_lr)
ece_lgb = expected_calibration_error(y_val.values, y_val_pred_lgb)

print(f"Expected Calibration Error (validation set):")
print(f"  WoE + LR:  {ece_lr:.4f}")
print(f"  LightGBM:  {ece_lgb:.4f}")

In [ ]:
def reliability_diagram(y_true, y_pred, n_bins=10, label="Model", ax=None):
    """Plot a reliability diagram (calibration curve) on the given axis."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))

    # Quantile-based bins so every bin has the same count
    bin_edges = np.quantile(y_pred, np.linspace(0, 1, n_bins + 1))
    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf
    bin_ids = np.digitize(y_pred, bin_edges[1:-1])

    bin_means_pred = []
    bin_means_actual = []
    bin_sizes = []
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        bin_means_pred.append(y_pred[mask].mean())
        bin_means_actual.append(y_true[mask].mean())
        bin_sizes.append(mask.sum())

    ax.plot([0, max(bin_means_pred) * 1.1], [0, max(bin_means_pred) * 1.1],
            color="black", linestyle="--", alpha=0.5, label="Perfect calibration")
    ax.scatter(bin_means_pred, bin_means_actual, s=80, label=label, zorder=3)
    ax.plot(bin_means_pred, bin_means_actual, alpha=0.7, zorder=2)

    ax.set_xlabel("Mean predicted PD")
    ax.set_ylabel("Actual default rate")
    ax.set_title(f"Reliability diagram: {label}")
    ax.legend()
    ax.grid(True, alpha=0.3)

    return ax


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
reliability_diagram(y_val.values, y_val_pred_lr, label=f"WoE+LR (ECE={ece_lr:.4f})", ax=axes[0])
reliability_diagram(y_val.values, y_val_pred_lgb, label=f"LightGBM (ECE={ece_lgb:.4f})", ax=axes[1])
plt.tight_layout()
plt.show()

**Read the reliability diagrams:**

- Points on the diagonal = perfectly calibrated at that score level
- Points below the diagonal = model overestimates risk (predicts higher PD than reality)
- Points above the diagonal = model underestimates risk

Wherever the curve drifts from the diagonal, isotonic regression will pull it back. Wherever the model is already calibrated, isotonic essentially leaves predictions unchanged.

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy.stats import ks_2samp


def ks_statistic(y_true, y_pred):
    return ks_2samp(y_pred[y_true == 1], y_pred[y_true == 0]).statistic


# Fit isotonic calibrator on validation predictions
iso_lgb = IsotonicRegression(out_of_bounds="clip")
iso_lgb.fit(y_val_pred_lgb, y_val.values)

# Apply to test predictions
y_test_pred_lgb_calibrated = iso_lgb.predict(y_test_pred_lgb)

# Also apply to validation for the reliability diagram comparison
y_val_pred_lgb_calibrated = iso_lgb.predict(y_val_pred_lgb)

print(f"Mean predicted PD (LightGBM, before): {y_val_pred_lgb.mean():.4f}")
print(f"Mean predicted PD (LightGBM, after):  {y_val_pred_lgb_calibrated.mean():.4f}")
print(f"Actual default rate (val):            {y_val.mean():.4f}")

In [ ]:
# ECE before and after
ece_lgb_after = expected_calibration_error(y_val.values, y_val_pred_lgb_calibrated)

# Ranking metrics should be unchanged
auc_before = roc_auc_score(y_val, y_val_pred_lgb)
auc_after = roc_auc_score(y_val, y_val_pred_lgb_calibrated)
ks_before = ks_statistic(y_val.values, y_val_pred_lgb)
ks_after = ks_statistic(y_val.values, y_val_pred_lgb_calibrated)
brier_before = brier_score_loss(y_val, y_val_pred_lgb)
brier_after = brier_score_loss(y_val, y_val_pred_lgb_calibrated)

print(f"LightGBM validation metrics, before/after isotonic calibration:")
print(f"  ECE:    {ece_lgb:.4f}  →  {ece_lgb_after:.4f}")
print(f"  AUC:    {auc_before:.4f}  →  {auc_after:.4f}")
print(f"  KS:     {ks_before:.4f}  →  {ks_after:.4f}")
print(f"  Brier:  {brier_before:.4f}  →  {brier_after:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
reliability_diagram(y_val.values, y_val_pred_lgb,
                    label=f"Before (ECE={ece_lgb:.4f})", ax=axes[0])
reliability_diagram(y_val.values, y_val_pred_lgb_calibrated,
                    label=f"After (ECE={ece_lgb_after:.4f})", ax=axes[1])
plt.suptitle("LightGBM calibration: before vs after isotonic regression", y=1.02)
plt.tight_layout()
plt.show()

**Section 3A takeaways:**

- Initial LightGBM ECE: 0.0054, after isotonic regression: 0.0000 — both well below the
  0.01 "meaningful miscalibration" threshold; calibration was already in good shape.
- The reliability diagram before calibration showed mild systematic underprediction across
  most of the score range (predicted PDs slightly lower than actual default rates) with a
  small overprediction at the top decile (PD ~0.28).
- After isotonic regression, the curve hugs the diagonal almost exactly.
- KS unchanged (isotonic preserves rank-order CDFs).
- AUC moved 0.7611 → 0.7628 (+0.0017): isotonic flattened some marginal non-monotonic
  miscalibration patterns, which has a small ranking benefit. The change is small but real.
- Brier improved slightly: 0.0677 → 0.0674.

**Why the calibration was already good:**

Two engineering choices from earlier milestones avoided the typical gradient-boosting
miscalibration issues:
1. No `scale_pos_weight` — class weighting in gradient boosting is the most common source
   of severe miscalibration. The natural 8% imbalance was fine for the training data size.
2. 215k training rows — calibration tends to be cleaner in large training sets because
   every score level has many examples.

The WoE+LR baseline was already well-calibrated (ECE = 0.0044), reflecting one of WoE
encoding's quiet strengths: bin-level WoE values are *defined* relative to the bad rate,
so the model produces honest probabilities by construction.

**Practical implication:** the LightGBM model now produces calibrated probabilities,
which is necessary for the threshold selection and fair lending analyses in Milestone 4:
- Pricing decisions tied to PD reflect real risk
- Expected loss calculations (PD × LGD × EAD) are unbiased  
- Thresholds chosen at specific PD values mean what they say

## Section 3B: Monotonic Constraints

Real production credit scorecards enforce monotonic relationships between key features and predicted default risk. This serves four purposes:

1. **Regulatory defensibility** — models with explainable, sensible feature-PD relationships are easier to document and defend in a regulatory review than models with arbitrary non-monotonic patterns.
2. **Fair lending robustness** — non-monotonic patterns can hide subtle proxies for protected classes.
3. **Stability over time** — constrained models are less likely to fit noise that won't generalize.
4. **Negligible AUC cost** — when the constraint aligns with domain reasoning, the model would have learned a similar shape anyway.

We constrain 8 features where the directional relationship is unambiguous from domain knowledge. We deliberately don't constrain features where the direction is ambiguous (`OCCUPATION_TYPE`, `income_log`) or where EDA showed a non-monotonic pattern (`bureau_count` — U-shaped).

**Note on age:** Age is partially protected under ECOA (over-62 protection) and the data shows a strong downward trend (2.58x default lift from 60+ vs 20-25). We include it with a monotonic constraint (older → lower PD) rather than dropping it — but flag this as a key fair-lending decision to revisit in Milestone 4. The constraint encodes the marginal direction holding other features fixed; interaction effects with employment, bureau, and income are still captured through LightGBM's tree structure.

In [ ]:
# Define the constraints
# Direction: -1 = higher feature → lower PD; +1 = higher feature → higher PD; 0 = unconstrained
constraint_directions = {
    "EXT_SOURCE_1": -1,
    "EXT_SOURCE_2": -1,
    "EXT_SOURCE_3": -1,
    "bureau_overdue_max": +1,
    "employment_years": -1,
    "age_years": -1,
    "payment_to_income": +1,
    "loan_to_income": +1,
}

# LightGBM expects monotone_constraints as a list in the same order as the feature columns
monotone_constraints = [
    constraint_directions.get(col, 0) for col in feature_cols
]

# Verify the mapping
print("Features with monotonic constraints:")
for col in feature_cols:
    direction = constraint_directions.get(col, 0)
    if direction != 0:
        arrow = "↑" if direction == 1 else "↓"
        print(f"  {col:<30} {arrow}")

unconstrained = [col for col in feature_cols if col not in constraint_directions]
print(f"\nUnconstrained features ({len(unconstrained)}): omitted for brevity")

In [ ]:
# Train the constrained LightGBM
# Everything is identical to the Milestone 2 model EXCEPT we add monotone_constraints

lgb_params_constrained = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "random_state": 42,
    "verbose": -1,
    "monotone_constraints": monotone_constraints,
    # monotone_constraints_method options: "basic" (fast, less accurate) or "intermediate" / "advanced"
    "monotone_constraints_method": "intermediate",
}

lgb_train_ds = lgb.Dataset(X_train_lgb, label=y_train, categorical_feature=categorical_cols)
lgb_val_ds = lgb.Dataset(X_val_lgb, label=y_val, categorical_feature=categorical_cols, reference=lgb_train_ds)

lgb_model_constrained = lgb.train(
    lgb_params_constrained,
    lgb_train_ds,
    num_boost_round=2000,
    valid_sets=[lgb_train_ds, lgb_val_ds],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=0),
    ],
)

print(f"Constrained model best iteration: {lgb_model_constrained.best_iteration}")

In [ ]:
# Evaluate the constrained model
y_val_pred_constrained = lgb_model_constrained.predict(X_val_lgb, num_iteration=lgb_model_constrained.best_iteration)

auc_constrained = roc_auc_score(y_val, y_val_pred_constrained)
ks_constrained = ks_statistic(y_val.values, y_val_pred_constrained)
brier_constrained = brier_score_loss(y_val, y_val_pred_constrained)

print("LightGBM comparison: unconstrained vs constrained\n")
print(f"{'Metric':<8} {'Unconstrained':>14} {'Constrained':>14} {'Δ':>10}")
print(f"{'AUC':<8} {auc_before:>14.4f} {auc_constrained:>14.4f} {auc_constrained - auc_before:>+10.4f}")
print(f"{'KS':<8} {ks_before:>14.4f} {ks_constrained:>14.4f} {ks_constrained - ks_before:>+10.4f}")
print(f"{'Brier':<8} {brier_before:>14.4f} {brier_constrained:>14.4f} {brier_constrained - brier_before:>+10.4f}")

In [ ]:
# Sanity check: for each constrained feature, the predicted PD should move monotonically
# with the feature value. Compare the constrained and unconstrained models on this.

def avg_pred_by_bin(X, y_pred, feature, n_bins=10):
    """Return mean prediction within each quantile bin of a feature."""
    sub = pd.DataFrame({feature: X[feature].values, "pred": y_pred})
    sub["bin"] = pd.qcut(sub[feature], n_bins, labels=False, duplicates="drop")
    return sub.groupby("bin")["pred"].mean()


fig, axes = plt.subplots(2, 4, figsize=(18, 8))
features_to_plot = list(constraint_directions.keys())

for i, feat in enumerate(features_to_plot):
    ax = axes[i // 4, i % 4]
    direction = constraint_directions[feat]
    
    # Unconstrained model
    pred_unc = avg_pred_by_bin(X_val_lgb, y_val_pred_lgb, feat)
    # Constrained model
    pred_con = avg_pred_by_bin(X_val_lgb, y_val_pred_constrained, feat)
    
    ax.plot(pred_unc.index, pred_unc.values, marker="o", label="Unconstrained", alpha=0.7)
    ax.plot(pred_con.index, pred_con.values, marker="s", label="Constrained", alpha=0.7)
    
    arrow = "↑" if direction == 1 else "↓"
    ax.set_title(f"{feat} (constraint {arrow})")
    ax.set_xlabel("Feature decile")
    ax.set_ylabel("Mean predicted PD")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Constrained vs unconstrained PD by feature decile", y=1.00)
plt.tight_layout()
plt.show()

**Section 3B takeaways:**

- AUC change with constraints: 0.7611 → 0.7611 (essentially zero). KS marginally improved (+0.0005). Brier marginally worse (+0.0002). Constraints had no meaningful metric cost on this dataset, likely because 215k training rows are enough for the model to learn monotonic patterns naturally where the signal is strong.
- The constraint plots show three patterns:
  - **EXT_SOURCEs**: smooth monotonic decline; constrained and unconstrained nearly identical. Constraint had nothing to do because the model already learned monotonic.
  - **age_years, employment_years**: mostly monotonic with small kinks in the unconstrained version that the constrained version smooths out.
  - **bureau_overdue_max, payment_to_income, loan_to_income**: bin-averaged plots show non-monotonic shapes even after constraint. This reflects an important subtlety — the monotonic constraint enforces direction *holding other features fixed*, not on the bin averages, which mix applicants with different other features. The visible non-monotonicity reflects selection effects in the data, not a constraint violation.

**Why constraints matter despite the small effect on this dataset:**

- Production credit models in regulated environments use monotonic constraints by default. Even when the unconstrained model would have learned similar shapes, the *guarantee* of monotonicity matters for regulatory documentation.
- For 4 of the 8 constrained features (the EXT_SOURCEs, `bureau_overdue_max`), the direction is unambiguous from domain reasoning — the constraint is essentially free.
- For age (a partially protected attribute), the constraint ensures the model's marginal use of age is in a single defensible direction holding other features equal. This is revisited in Milestone 4 fair lending analysis.
- For payment-to-income and loan-to-income ratios, the constraint ensures the model behaves like an underwriter at the individual level — higher burden → higher risk for any otherwise-identical applicant. The aggregate decile plots reflect selection effects, not constraint failures.

**The constrained LightGBM is the production model going forward.** Milestone 4 uses calibrated predictions from this constrained model for threshold selection, fair lending, and SHAP analysis.

In [ ]:
# The production model: constrained LightGBM with isotonic calibration

# Refit isotonic calibrator on the constrained model's val predictions
iso_lgb_constrained = IsotonicRegression(out_of_bounds="clip")
iso_lgb_constrained.fit(y_val_pred_constrained, y_val.values)

# Calibrated predictions on validation and test
y_val_pred_final = iso_lgb_constrained.predict(y_val_pred_constrained)
y_test_pred_constrained = lgb_model_constrained.predict(
    X_test_lgb, num_iteration=lgb_model_constrained.best_iteration
)
y_test_pred_final = iso_lgb_constrained.predict(y_test_pred_constrained)

# Full metrics on validation
auc_final = roc_auc_score(y_val, y_val_pred_final)
ks_final = ks_statistic(y_val.values, y_val_pred_final)
brier_final = brier_score_loss(y_val, y_val_pred_final)
ece_final = expected_calibration_error(y_val.values, y_val_pred_final)

print(f"Final production model (constrained + calibrated LightGBM):")
print(f"  AUC:    {auc_final:.4f}")
print(f"  KS:     {ks_final:.4f}")
print(f"  Brier:  {brier_final:.4f}")
print(f"  ECE:    {ece_final:.4f}")
print(f"  Mean predicted PD (val):    {y_val_pred_final.mean():.4f}")
print(f"  Actual default rate (val):  {y_val.mean():.4f}")

# Save predictions for Milestone 4
import pickle
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

predictions = {
    "y_val": y_val.values,
    "y_test": y_test.values,
    "y_val_pred_lr": y_val_pred_lr,
    "y_val_pred_final": y_val_pred_final,
    "y_test_pred_lr": y_test_pred_lr,
    "y_test_pred_final": y_test_pred_final,
    "test_idx": test_idx,
    "val_idx": val_idx,
}

with open(processed_dir / "predictions.pkl", "wb") as f:
    pickle.dump(predictions, f)

print(f"\nSaved final production predictions for Milestone 4 to {processed_dir / 'predictions.pkl'}")

## Section 3C: Segment Model for Thin-File

The thin-file segment (`bureau_count <= 1`, 26% of the book, 9.43% default rate) is 
structurally different from the rest of the population — these applicants are largely 
invisible to bureau-based features. A pooled model learns patterns dominated by the 
74% of applicants with bureau history, which may or may not generalize to thin-file.

Three modeling strategies are possible:
- **Pooled with segment-aware features (current production)**: include `is_thin_file` 
  in the feature set, let the model use it implicitly. This is what the 3B model does.
- **Explicit segment model (this section)**: train a separate model on only thin-file 
  rows, apply to thin-file applicants at scoring time.
- **Hierarchical model**: too complex for this project; mentioned only for completeness.

This section trains the explicit segment model and compares it against the pooled 
production model, evaluated on the thin-file subset of validation. The interesting 
question is whether specialization beats data volume.

In [ ]:
# Identify thin-file rows in each split
is_thin_file_train = train["is_thin_file"] == 1
is_thin_file_val = val["is_thin_file"] == 1
is_thin_file_test = test["is_thin_file"] == 1

print(f"Thin-file fraction in each split:")
print(f"  Train: {is_thin_file_train.mean():.2%} ({is_thin_file_train.sum():,} of {len(train):,})")
print(f"  Val:   {is_thin_file_val.mean():.2%} ({is_thin_file_val.sum():,} of {len(val):,})")
print(f"  Test:  {is_thin_file_test.mean():.2%} ({is_thin_file_test.sum():,} of {len(test):,})")

# Thin-file subsets of the LightGBM-formatted features
X_train_thin = X_train_lgb[is_thin_file_train.values]
y_train_thin = y_train[is_thin_file_train.values]
X_val_thin = X_val_lgb[is_thin_file_val.values]
y_val_thin = y_val[is_thin_file_val.values]

print(f"\nThin-file segment default rate (train): {y_train_thin.mean():.4f}")
print(f"Thin-file segment default rate (val):   {y_val_thin.mean():.4f}")

In [ ]:
# Train a LightGBM on only the thin-file rows
# Use the same constraints as the pooled model — same regulatory rationale applies
# Drop is_thin_file from features (it's constant=1 in this subset, contributes nothing)

# Identify is_thin_file's position in feature_cols
thin_file_idx = feature_cols.index("is_thin_file")
segment_feature_cols = [c for c in feature_cols if c != "is_thin_file"]
segment_monotone = [monotone_constraints[i] for i, c in enumerate(feature_cols) if c != "is_thin_file"]

X_train_thin_seg = X_train_thin.drop(columns=["is_thin_file"])
X_val_thin_seg = X_val_thin.drop(columns=["is_thin_file"])

lgb_params_segment = {
    **lgb_params_constrained,
    "monotone_constraints": segment_monotone,
}

lgb_train_thin_ds = lgb.Dataset(
    X_train_thin_seg, label=y_train_thin, categorical_feature=categorical_cols
)
lgb_val_thin_ds = lgb.Dataset(
    X_val_thin_seg, label=y_val_thin, categorical_feature=categorical_cols, reference=lgb_train_thin_ds
)

lgb_segment_model = lgb.train(
    lgb_params_segment,
    lgb_train_thin_ds,
    num_boost_round=2000,
    valid_sets=[lgb_train_thin_ds, lgb_val_thin_ds],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=0),
    ],
)

print(f"Segment model best iteration: {lgb_segment_model.best_iteration}")
print(f"Training rows used: {len(X_train_thin_seg):,}")

In [ ]:
# Predict with the SEGMENT model on the thin-file validation rows
y_val_thin_pred_segment_raw = lgb_segment_model.predict(
    X_val_thin_seg, num_iteration=lgb_segment_model.best_iteration
)

# Calibrate the segment model using thin-file validation rows
iso_segment = IsotonicRegression(out_of_bounds="clip")
iso_segment.fit(y_val_thin_pred_segment_raw, y_val_thin.values)
y_val_thin_pred_segment = iso_segment.predict(y_val_thin_pred_segment_raw)

# Predict with the POOLED constrained model on the same thin-file validation rows
y_val_thin_pred_pooled_raw = lgb_model_constrained.predict(
    X_val_lgb[is_thin_file_val.values], num_iteration=lgb_model_constrained.best_iteration
)
# Apply the pooled model's existing calibrator
y_val_thin_pred_pooled = iso_lgb_constrained.predict(y_val_thin_pred_pooled_raw)

# Predict with LR baseline on thin-file validation rows
y_val_thin_pred_lr = y_val_pred_lr[is_thin_file_val.values]

print(f"All three sets of predictions ready for thin-file segment ({is_thin_file_val.sum():,} rows)")

In [ ]:
# Evaluate all three models on the thin-file segment of validation
results = []
for name, preds in [
    ("LR baseline (pooled)", y_val_thin_pred_lr),
    ("LightGBM (pooled, constrained+calibrated)", y_val_thin_pred_pooled),
    ("LightGBM (thin-file segment)", y_val_thin_pred_segment),
]:
    results.append({
        "Model": name,
        "AUC": roc_auc_score(y_val_thin, preds),
        "KS": ks_statistic(y_val_thin.values, preds),
        "Brier": brier_score_loss(y_val_thin, preds),
        "Mean PD": preds.mean(),
    })

results_df = pd.DataFrame(results)
print(f"Performance on thin-file validation ({is_thin_file_val.sum():,} rows, default rate {y_val_thin.mean():.4f}):\n")
print(results_df.to_string(index=False))

In [ ]:
# For the same thin-file applicants, do the two LightGBM models give similar PD predictions?
# A high correlation means they're learning similar patterns; a low correlation means real specialization.

correlation = np.corrcoef(y_val_thin_pred_pooled, y_val_thin_pred_segment)[0, 1]
mean_abs_diff = np.abs(y_val_thin_pred_pooled - y_val_thin_pred_segment).mean()

print(f"Correlation between pooled and segment predictions on thin-file: {correlation:.4f}")
print(f"Mean absolute prediction difference: {mean_abs_diff:.4f}")
print(f"Median absolute prediction difference: {np.median(np.abs(y_val_thin_pred_pooled - y_val_thin_pred_segment)):.4f}")

# Plot the comparison
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_val_thin_pred_pooled, y_val_thin_pred_segment, alpha=0.2, s=10)
ax.plot([0, 0.5], [0, 0.5], color="red", linestyle="--", alpha=0.5, label="Identical predictions")
ax.set_xlabel("Pooled model PD")
ax.set_ylabel("Segment model PD")
ax.set_title("Pooled vs segment predictions on thin-file applicants")
ax.legend()
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 0.5)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# What features does the segment model lean on most?
segment_importance = pd.DataFrame({
    "feature": X_train_thin_seg.columns,
    "importance": lgb_segment_model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)

print("Top 15 features from segment model (by gain):")
print(segment_importance.head(15))

# Compare to pooled model importance
pooled_importance = pd.DataFrame({
    "feature": X_train_lgb.columns,
    "importance": lgb_model_constrained.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)

# Side-by-side top 10
comparison = pd.merge(
    pooled_importance.head(15).reset_index(drop=True).reset_index().rename(columns={"index": "pooled_rank", "feature": "pooled_feature"}),
    segment_importance.head(15).reset_index(drop=True).reset_index().rename(columns={"index": "segment_rank", "feature": "segment_feature"}),
    left_index=True, right_index=True,
)[["pooled_rank", "pooled_feature", "segment_rank", "segment_feature"]]
print("\nTop 15 features: pooled vs segment, side by side")
print(comparison.to_string(index=False))

**Section 3C takeaways:**

- Segment model trained on 56,281 thin-file rows; pooled model trained on 215,257 rows total.
- Performance on thin-file validation (11,904 rows, default rate 9.27%):
  - Pooled LightGBM (constrained + calibrated): AUC 0.7405, KS 0.3475, Brier 0.0775
  - Segment LightGBM (constrained + calibrated): AUC 0.7401, KS 0.3488, Brier 0.0777
  - LR baseline: AUC 0.7224, KS 0.3206, Brier 0.0789
- Differences between pooled and segment models are within noise on all three metrics.
- Pooled vs segment prediction correlation: 0.9345. Mean absolute prediction difference: 0.020. The two models largely agree on each applicant.

**Feature importance comparison reveals what the pooled model is implicitly doing:**

Most features rank similarly in both models, but the segment model elevates "soft" identifying signals:
- `DAYS_ID_PUBLISH` jumps from rank 10 (pooled) to rank 2 (segment)
- `DAYS_REGISTRATION` jumps from rank 12 to rank 6
- `REGION_POPULATION_RELATIVE` enters the segment top 15 but is not in the pooled top 15

This makes sense: for applicants without bureau history, the model has to lean on alternative signals — civic engagement proxies (ID/registration recency), geographic context, and the same EXT_SOURCE_2 that everyone has. The pooled model is already doing this internally for thin-file applicants via its tree structure; the segment model just makes the pattern visible.

**Verdict: explicit segment modeling is NOT worth the operational complexity.**

Reasoning:
- No meaningful metric improvement (differences within noise on AUC, KS, Brier).
- The pooled model is already segment-aware via the `is_thin_file` feature and LightGBM's native NaN handling. Trees can route thin-file applicants through different sub-paths without a separate model.
- EXT_SOURCE_2's near-universal coverage (99.8%) means thin-file applicants are not starved of signal.
- Two models in production means two versions to maintain, monitor for drift, and explain to regulators. The cost is real; the benefit is not.

**The segment-vs-pooled exercise is still valuable as a diagnostic.** It revealed what features the model implicitly uses for thin-file applicants (`DAYS_ID_PUBLISH`, `DAYS_REGISTRATION`), which is useful context for fair lending analysis and for stakeholder conversations about why the model makes specific decisions.

**The pooled constrained-and-calibrated LightGBM remains the production model** for Milestone 4 (threshold selection, fair lending, adverse action analysis).